# Description

In this notebook, we extract the latex formulas from the learned CEQL model that was trained to approximate FR function.

In [1]:
import sympy as sp
import torch
import dill
from pathlib import Path
# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------
SAVE_PATH = Path("models/frf_model_full.pt")
device = 'cpu'

ckpt = torch.load(
    SAVE_PATH,
    map_location=device,
    pickle_module=dill,
    weights_only=False,
)

model = ckpt["model"].to(device)
model.eval()

# ------------------------------------------------------------
# symbolic variables
# ------------------------------------------------------------
f = sp.Symbol("f")
sin_phi = sp.Symbol(r"\sin{\phi}")
cos_phi = sp.Symbol(r"\cos{\phi}")
mu = sp.Symbol(r"\boldsymbol{\mu}")

# ------------------------------------------------------------
# helper
# ------------------------------------------------------------
def coef(w: torch.Tensor, decimals: int = 2):
    val = float(w.real.detach().cpu().item())
    return round(val, decimals)

# ------------------------------------------------------------
# build expression
# ------------------------------------------------------------
a = coef(model.a)
b = coef(model.b)

expr = a * f + b

for i, block in enumerate(model.resonators, start=1):

    c = coef(block.c)
    d = coef(block.d)
    g = coef(block.g)

    g_i = sp.Symbol(f"g_{i}")

    expr = expr + c / ((d * f - g_i) ** 2 + g)

# ------------------------------------------------------------
# equation
# ------------------------------------------------------------
Hhat = sp.Symbol(r"\hat{H}")
lhs = sp.Function(Hhat)(f, sin_phi, cos_phi, mu)

eq = sp.Eq(lhs, expr)

# plain output
print(eq)

# LaTeX output
latex_str = sp.latex(eq)

print("\nLaTeX:\n")
print(r"\begin{equation}")
print(latex_str)
print(r"\end{equation}")

Eq(\hat{H}(f, \sin{\phi}, \cos{\phi}, \boldsymbol{\mu}), 2.55*f + 0.34 + 0.13/((1.58*f - g_6)**2 - 0.35) + 0.18/((-0.06*f - g_4)**2 + 0.59) - 1.16/((-0.06*f - g_2)**2 + 0.81) + 0.75/((-0.29*f - g_1)**2 + 0.47) - 0.03/(-0.14*f - g_3)**2 - 0.47/(-1.46*f - g_5)**2)

LaTeX:

\begin{equation}
\hat{H}{\left(f,\sin{\phi},\cos{\phi},\boldsymbol{\mu} \right)} = 2.55 f + 0.34 + \frac{0.13}{\left(1.58 f - g_{6}\right)^{2} - 0.35} + \frac{0.18}{\left(- 0.06 f - g_{4}\right)^{2} + 0.59} - \frac{1.16}{\left(- 0.06 f - g_{2}\right)^{2} + 0.81} + \frac{0.75}{\left(- 0.29 f - g_{1}\right)^{2} + 0.47} - \frac{0.03}{\left(- 0.14 f - g_{3}\right)^{2}} - \frac{0.47}{\left(- 1.46 f - g_{5}\right)^{2}}
\end{equation}


In [3]:
eq

Eq(\hat{H}(f, \sin{\phi}, \cos{\phi}, \boldsymbol{\mu}), 2.55*f + 0.34 + 0.13/((1.58*f - g_6)**2 - 0.35) + 0.18/((-0.06*f - g_4)**2 + 0.59) - 1.16/((-0.06*f - g_2)**2 + 0.81) + 0.75/((-0.29*f - g_1)**2 + 0.47) - 0.03/(-0.14*f - g_3)**2 - 0.47/(-1.46*f - g_5)**2)